# <b> Data Processing and Analysis of Austin Animal Center Data </b>

#### TO DOs
- have to reconcile multiple id entries for the same animal (e.g. A721033) in the intakes and outcomes dataset (they match up between datasets)
- Lookup outcome types for the outcome dataset
- For repeat animals, do we want to consolidate so that they don't pollute potential outcome predictions (small sample compared to all data)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df_outcomes = pd.read_csv('../data/outcomes.csv')
df_intakes = pd.read_csv('../data/intakes.csv')

## Analysis

In [3]:
print("Intake dataset shape:", df_intakes.shape)
print("Outcomes dataset shape:", df_outcomes.shape)
# 37 more entries in the intakes dataset (Spoiler: it's duplicate entries)

Intake dataset shape: (173812, 12)
Outcomes dataset shape: (173775, 12)


In [69]:
df_intakes.head()

,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
0,A521520,Nina,10/01/2013 07:51:00 AM,October 2013,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,7 years,Border Terrier/Border Collie,White/Tan
1,A664235,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
2,A664236,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
3,A664237,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
4,A664233,Stevie,10/01/2013 08:53:00 AM,October 2013,7405 Springtime in Austin (TX),Stray,Injured,Dog,Intact Female,3 years,Pit Bull Mix,Blue/White


In [70]:
df_intakes.info()
# the name column seems to be the only one with missing values


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173812 entries, 0 to 173811
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Animal ID         173812 non-null  object
 1   Name              123821 non-null  object
 2   DateTime          173812 non-null  object
 3   MonthYear         173812 non-null  object
 4   Found Location    173812 non-null  object
 5   Intake Type       173812 non-null  object
 6   Intake Condition  173812 non-null  object
 7   Animal Type       173812 non-null  object
 8   Sex upon Intake   173811 non-null  object
 9   Age upon Intake   173812 non-null  object
 10  Breed             173812 non-null  object
 11  Color             173812 non-null  object
dtypes: object(12)
memory usage: 15.9+ MB


In [71]:
# check if there are any other missing values
print(df_intakes.isnull().sum())

Animal ID               0
Name                49991
DateTime                0
MonthYear               0
Found Location          0
Intake Type             0
Intake Condition        0
Animal Type             0
Sex upon Intake         1
Age upon Intake         0
Breed                   0
Color                   0
dtype: int64


In [72]:
# all columns are object type, so we will need to convert some of them to datetime and categorical types
df_intakes.dtypes

Animal ID           object
Name                object
DateTime            object
MonthYear           object
Found Location      object
Intake Type         object
Intake Condition    object
Animal Type         object
Sex upon Intake     object
Age upon Intake     object
Breed               object
Color               object
dtype: object

## Animal IDs

In [73]:
# how many unique animals do we have?
print("Number of unique animalIDs in the intakes dataset:", df_intakes['Animal ID'].nunique())
print("Number of total records in the intakes dataset:", df_intakes.shape[0])

# have to be careful here because what if some ID's are repeated in the intake on accident
# maybe repeat ids are due to multiple intakes for the same animal, so we will check if there are any duplicate records
# and maybe the return of some animals to the shelter is causing this too

Number of unique animalIDs in the intakes dataset: 156287
Number of total records in the intakes dataset: 173812


In [74]:
# check dupelicate records
duplicate_records = df_intakes[df_intakes.duplicated()]

print("Number of duplicate records:", duplicate_records.shape[0])

duplicate_records.head()

Number of duplicate records: 36


,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
12537,A681446,NaN,06/16/2014 04:38:00 PM,June 2014,6208 Tupelo Dr in Austin (TX),Wildlife,Normal,Other,Unknown,1 year,Bat,Brown
13177,A668183,Nacha,06/27/2014 11:34:00 AM,June 2014,Austin (TX),Public Assist,Normal,Dog,Spayed Female,10 months,Dachshund Mix,Black/White
23682,A695798,Jim,01/23/2015 11:48:00 AM,January 2015,Austin (TX),Owner Surrender,Sick,Cat,Neutered Male,10 months,Domestic Shorthair Mix,Orange
24370,A696688,Mari,02/10/2015 11:00:00 AM,February 2015,Austin (TX),Owner Surrender,Normal,Cat,Spayed Female,3 years,Domestic Shorthair Mix,Calico/White
25933,A698864,NaN,03/19/2015 01:37:00 PM,March 2015,12129 Glass Rd in Travis (TX),Wildlife,Injured,Other,Unknown,1 year,Skunk Mix,Black


In [75]:
df_intakes[df_intakes["Animal ID"] == "A681446"]

,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
12536,A681446,NaN,06/16/2014 04:38:00 PM,June 2014,6208 Tupelo Dr in Austin (TX),Wildlife,Normal,Other,Unknown,1 year,Bat,Brown
12537,A681446,NaN,06/16/2014 04:38:00 PM,June 2014,6208 Tupelo Dr in Austin (TX),Wildlife,Normal,Other,Unknown,1 year,Bat,Brown


In [76]:
# drop duplicate records
df_intakes = df_intakes.drop_duplicates()

In [ ]:
df_intakes[df_intakes.duplicated()]
# no more dupes

,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color


In [86]:
# okay only 36 entires are duplicates (now dropped)

# lets check some repeated IDs to see if they're being reused or if they are being returned to the shelter
print(df_intakes["Animal ID"].value_counts().head(10))

Animal ID
A721033    33
A718223    14
A718877    12
A705625    12
A706536    11
A761266    10
A700407     9
A737814     9
A716018     9
A717053     9
Name: count, dtype: int64


In [78]:
print(df_intakes[df_intakes["Animal ID"] == "A721033"].shape)
df_intakes[df_intakes["Animal ID"] == "A721033"].head(10)
# one dog repeated entries, have to check if matches up with the outcomes dataset

(33, 12)


,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
43498,A721033,Lil Bit,02/20/2016 10:44:00 AM,February 2016,2508 E Riverside Dr in Austin (TX),Stray,Normal,Dog,Neutered Male,9 months,Rat Terrier Mix,Tricolor/Brown Brindle
51001,A721033,Lil Bit,07/10/2016 11:53:00 AM,July 2016,6317 E Ben White Blvd in Austin (TX),Stray,Normal,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
53835,A721033,Lil Bit,09/03/2016 02:30:00 PM,September 2016,6010 E Riverside in Austin (TX),Public Assist,Normal,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
56121,A721033,Lil Bit,10/20/2016 10:47:00 PM,October 2016,6200 E. Ben White Blvd in Austin (TX),Public Assist,Normal,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
58575,A721033,Lil Bit,12/15/2016 10:07:00 AM,December 2016,S Pleasant Valley Rd And E Riverside Dr in Aus...,Public Assist,Normal,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
59568,A721033,Lil Bit,01/09/2017 02:26:00 PM,January 2017,6210 E Ben White Blvd in Austin (TX),Stray,Injured,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
60211,A721033,Lil Bit,01/26/2017 06:55:00 AM,January 2017,901 W Ben White Blvd in Austin (TX),Public Assist,Normal,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
60407,A721033,Lil Bit,01/30/2017 11:05:00 PM,January 2017,6210 E Ben White in Austin (TX),Public Assist,Normal,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
60647,A721033,Lil Bit,02/06/2017 10:13:00 AM,February 2017,6210 E Ben White in Austin (TX),Public Assist,Normal,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
62261,A721033,Lil Bit,03/15/2017 09:24:00 AM,March 2017,E Ben White Blvd & Montopolis Dr in Austin (TX),Public Assist,Normal,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle


In [79]:
print(df_outcomes[df_outcomes["Animal ID"] == "A721033"].shape)
df_outcomes[df_outcomes["Animal ID"] == "A721033"].head(10)

# yeah it matches up exactly with intake IDs so that's good
# we'll have to decide what to do with duplicate entries (TODO)

(33, 12)


,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
45568,A721033,2015-05-20,Lil Bit,2016-02-20T16:18:00,02-2016,Return to Owner,NaN,Dog,Neutered Male,9 months,Rat Terrier Mix,Tricolor/Brown Brindle
52371,A721033,2015-05-20,Lil Bit,2016-07-11T17:44:00,07-2016,Return to Owner,NaN,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
55434,A721033,2015-05-20,Lil Bit,2016-09-12T13:40:00,09-2016,Return to Owner,NaN,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
57306,A721033,2015-05-20,Lil Bit,2016-10-21T18:55:00,10-2016,Return to Owner,NaN,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
59742,A721033,2015-05-20,Lil Bit,2016-12-16T11:32:00,12-2016,Return to Owner,NaN,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
60834,A721033,2015-05-20,Lil Bit,2017-01-10T16:20:00,01-2017,Return to Owner,NaN,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
61554,A721033,2015-05-20,Lil Bit,2017-01-28T15:22:00,01-2017,Return to Owner,NaN,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
61789,A721033,2015-05-20,Lil Bit,2017-02-02T11:19:00,02-2017,Return to Owner,NaN,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
62014,A721033,2015-05-20,Lil Bit,2017-02-07T17:26:00,02-2017,Return to Owner,NaN,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle
63623,A721033,2015-05-20,Lil Bit,2017-03-16T12:50:00,03-2017,Return to Owner,NaN,Dog,Neutered Male,1 year,Rat Terrier Mix,Tricolor/Brown Brindle


In [80]:
# and a differnt ID number
print(df_intakes[df_intakes["Animal ID"] == "A718223"].shape)
df_intakes[df_intakes["Animal ID"] == "A718223"]

(14, 12)


,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
41303,A718223,Bubba,12/25/2015 11:41:00 AM,December 2015,E 6Th St And Trinity in Austin (TX),Public Assist,Normal,Dog,Neutered Male,7 months,St. Bernard Rough Coat/Boxer,White/Brown
41435,A718223,Bubba,12/29/2015 09:24:00 AM,December 2015,6306 E. Riverside in Austin (TX),Public Assist,Normal,Dog,Neutered Male,7 months,St. Bernard Rough Coat/Boxer,White/Brown
44383,A718223,Bubba,03/10/2016 08:38:00 PM,March 2016,7010 W Sh 71 in Austin (TX),Public Assist,Normal,Dog,Neutered Male,9 months,St. Bernard Rough Coat/Boxer,White/Brown
47059,A718223,Bubba,05/03/2016 10:52:00 AM,May 2016,2606 Wheless Ln in Austin (TX),Public Assist,Normal,Dog,Neutered Male,11 months,St. Bernard Rough Coat/Boxer,White/Brown
58814,A718223,Bubba,12/20/2016 05:52:00 PM,December 2016,500 E 7Th St in Austin (TX),Public Assist,Normal,Dog,Neutered Male,1 year,St. Bernard Rough Coat/Boxer,White/Brown
59878,A718223,Bubba,01/18/2017 01:31:00 PM,January 2017,Austin (TX),Public Assist,Normal,Dog,Neutered Male,1 year,St. Bernard Rough Coat/Boxer,White/Brown
70885,A718223,Bubba,08/22/2017 04:12:00 PM,August 2017,500 E 7Th in Austin (TX),Public Assist,Normal,Dog,Neutered Male,2 years,St. Bernard Rough Coat/Boxer,White/Brown
72113,A718223,Bubba,09/12/2017 02:11:00 PM,September 2017,Austin (TX),Public Assist,Normal,Dog,Neutered Male,2 years,St. Bernard Rough Coat/Boxer,White/Brown
75758,A718223,Bubba,12/04/2017 12:39:00 AM,December 2017,Congress And William Cannon in Austin (TX),Public Assist,Normal,Dog,Neutered Male,2 years,St. Bernard Rough Coat/Boxer,White/Brown
76967,A718223,Bubba,01/06/2018 12:39:00 PM,January 2018,Austin (TX),Owner Surrender,Normal,Dog,Neutered Male,2 years,St. Bernard Rough Coat/Boxer,White/Brown


In [81]:
print(df_outcomes[df_outcomes["Animal ID"] == "A718223"].shape)
df_outcomes[df_outcomes["Animal ID"] == "A718223"].head(10)

(14, 12)


,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
43385,A718223,2015-05-24,Bubba,2015-12-26T13:52:00,12-2015,Return to Owner,NaN,Dog,Neutered Male,7 months,St. Bernard Rough Coat/Boxer,White/Brown
44115,A718223,2015-05-24,Bubba,2016-01-12T15:35:00,01-2016,Return to Owner,NaN,Dog,Neutered Male,7 months,St. Bernard Rough Coat/Boxer,White/Brown
46424,A718223,2015-05-24,Bubba,2016-03-11T16:22:00,03-2016,Return to Owner,NaN,Dog,Neutered Male,9 months,St. Bernard Rough Coat/Boxer,White/Brown
48745,A718223,2015-05-24,Bubba,2016-05-04T15:28:00,05-2016,Return to Owner,NaN,Dog,Neutered Male,11 months,St. Bernard Rough Coat/Boxer,White/Brown
60441,A718223,2015-05-24,Bubba,2016-12-31T16:49:00,12-2016,Return to Owner,NaN,Dog,Neutered Male,1 year,St. Bernard Rough Coat/Boxer,White/Brown
61397,A718223,2015-05-24,Bubba,2017-01-24T14:24:00,01-2017,Return to Owner,NaN,Dog,Neutered Male,1 year,St. Bernard Rough Coat/Boxer,White/Brown
71798,A718223,2015-05-24,Bubba,2017-08-25T11:24:00,08-2017,Return to Owner,NaN,Dog,Neutered Male,2 years,St. Bernard Rough Coat/Boxer,White/Brown
73025,A718223,2015-05-24,Bubba,2017-09-20T17:30:00,09-2017,Return to Owner,NaN,Dog,Neutered Male,2 years,St. Bernard Rough Coat/Boxer,White/Brown
76519,A718223,2015-05-24,Bubba,2017-12-08T16:25:00,12-2017,Return to Owner,NaN,Dog,Neutered Male,2 years,St. Bernard Rough Coat/Boxer,White/Brown
77670,A718223,2015-05-24,Bubba,2018-01-06T15:55:00,01-2018,Return to Owner,NaN,Dog,Neutered Male,2 years,St. Bernard Rough Coat/Boxer,White/Brown


In [6]:
df_outcomes["Breed"].value_counts()

Breed
Domestic Shorthair Mix                  34039
Domestic Shorthair                      23842
Pit Bull Mix                            10151
Labrador Retriever Mix                   8812
Chihuahua Shorthair Mix                  6920
                                        ...  
Whippet/Anatol Shepherd                     1
Yorkshire Terrier/Australian Terrier        1
Chihuahua Shorthair/Shiba Inu               1
Beagle/Cairn Terrier                        1
Pekingese/Lhasa Apso                        1
Name: count, Length: 3010, dtype: int64

### Looking at Outcomes Dataset

In [82]:
print(df_intakes.columns)

print(df_outcomes.columns)

Index(['Animal ID', 'Name', 'DateTime', 'MonthYear', 'Found Location',
       'Intake Type', 'Intake Condition', 'Animal Type', 'Sex upon Intake',
       'Age upon Intake', 'Breed', 'Color'],
      dtype='object')
Index(['Animal ID', 'Date of Birth', 'Name', 'DateTime', 'MonthYear',
       'Outcome Type', 'Outcome Subtype', 'Animal Type', 'Sex upon Outcome',
       'Age upon Outcome', 'Breed', 'Color'],
      dtype='object')


In [83]:
df_outcomes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173775 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Animal ID         173775 non-null  object
 1   Date of Birth     173775 non-null  object
 2   Name              123991 non-null  object
 3   DateTime          173775 non-null  object
 4   MonthYear         173775 non-null  object
 5   Outcome Type      173729 non-null  object
 6   Outcome Subtype   79660 non-null   object
 7   Animal Type       173775 non-null  object
 8   Sex upon Outcome  173774 non-null  object
 9   Age upon Outcome  173766 non-null  object
 10  Breed             173775 non-null  object
 11  Color             173775 non-null  object
dtypes: object(12)
memory usage: 15.9+ MB


In [ ]:
print(df_outcomes["Outcome Type"].value_counts())
print()
print("Number of outcome types:", df_outcomes["Outcome Type"].nunique())
# TODO: lookup what each outcome type means, and maybe group them into categories for the model

# maybe do top 5 outcome types

Outcome Type
Adoption           84598
Transfer           48689
Return to Owner    25691
Euthanasia         10833
Died                1672
Rto-Adopt           1241
Disposal             877
Missing               92
Relocate              29
Stolen                 5
Lost                   2
Name: count, dtype: int64

Number of outcome types: 11


In [87]:
# how many breeds are in outcomes
df_outcomes["Breed"].nunique()

3010

In [88]:
df_outcomes.head()

,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green
3,A679066,2014-04-16,NaN,2014-05-16T00:00:00-05:00,05-2014,NaN,NaN,Other,Unknown,4 weeks,Rabbit Sh,Brown
4,A680855,2014-05-25,NaN,2014-06-10T00:00:00-05:00,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black


In [ ]:
# any duplicate records in the outcomes dataset?
duplicate_records_outcomes = df_outcomes[df_outcomes.duplicated()]
duplicate_records_outcomes.head()

,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
1704,A755088,2016-09-29,Machete,2017-08-19T00:00:00-05:00,08-2017,Adoption,NaN,Dog,Spayed Female,10 months,Australian Kelpie Mix,Black/White
4154,A660948,2013-07-08,*Roy,2013-11-03T18:16:00,11-2013,Adoption,NaN,Cat,Neutered Male,3 months,Domestic Shorthair Mix,Brown Tabby/White
15699,A682781,2013-01-02,NaN,2014-07-03T09:00:00,07-2014,Transfer,SCRP,Cat,Neutered Male,1 year,Domestic Shorthair Mix,White/Black
16655,A683782,2014-02-15,NaN,2014-07-16T09:00:00,07-2014,Transfer,SCRP,Cat,Neutered Male,4 months,Domestic Shorthair Mix,Brown Tabby
19233,A686827,2012-02-27,NaN,2014-08-28T09:00:00,08-2014,Transfer,SCRP,Cat,Spayed Female,2 years,Domestic Shorthair Mix,Calico


In [93]:
# drop duplicate records
df_outcomes = df_outcomes.drop_duplicates()

In [ ]:
df_outcomes.shape
df_intakes.shape

(173749, 12)

In [95]:
df_intakes.shape

(173776, 12)

In [ ]:
# the two datasets are going to have a differing number of records because 
# the data collection ends at the same time in the intakes dataset so 
# the outcomes dataset will have not yet processed those new intakes or doesn't have outcomes for them yet
# So the intakes dataset will have more records than the outcomes dataset.

In [107]:
print(df_outcomes.dtypes)
print("----------------")
print(df_intakes.dtypes)

Animal ID           object
Date of Birth       object
Name                object
DateTime            object
MonthYear           object
Outcome Type        object
Outcome Subtype     object
Animal Type         object
Sex upon Outcome    object
Age upon Outcome    object
Breed               object
Color               object
dtype: object
----------------
Animal ID           object
Name                object
DateTime            object
MonthYear           object
Found Location      object
Intake Type         object
Intake Condition    object
Animal Type         object
Sex upon Intake     object
Age upon Intake     object
Breed               object
Color               object
dtype: object


In [ ]:
# I don't think name will matter nor be useful for the model, so we will drop it from the outcomes/intakes dataset
# we should consolidate dateTime and MonthYear

## Merging Datasets

In [ ]:
# lets match the two datasets on the Animal ID column to see how many records match up, not many to many
merged_df = pd.merge(df_intakes, df_outcomes, on='Animal ID', how='inner')

merged_df.shape

(221122, 23)

In [100]:
merged_df.head()

,Animal ID,Name_x,DateTime_x,MonthYear_x,Found Location,Intake Type,Intake Condition,Animal Type_x,Sex upon Intake,Age upon Intake,...,Name_y,DateTime_y,MonthYear_y,Outcome Type,Outcome Subtype,Animal Type_y,Sex upon Outcome,Age upon Outcome,Breed_y,Color_y
0,A521520,Nina,10/01/2013 07:51:00 AM,October 2013,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,7 years,...,Nina,2013-10-01T15:39:00,10-2013,Return to Owner,NaN,Dog,Spayed Female,7 years,Border Terrier/Border Collie,White/Tan
1,A664235,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,...,NaN,2013-10-01T10:39:00,10-2013,Transfer,Partner,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
2,A664236,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,...,NaN,2013-10-01T10:44:00,10-2013,Transfer,Partner,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
3,A664237,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,...,NaN,2013-10-01T10:44:00,10-2013,Transfer,Partner,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
4,A664233,Stevie,10/01/2013 08:53:00 AM,October 2013,7405 Springtime in Austin (TX),Stray,Injured,Dog,Intact Female,3 years,...,Stevie,2013-10-01T15:33:00,10-2013,Euthanasia,Suffering,Dog,Intact Female,3 years,Pit Bull Mix,Blue/White


### AI Outputs (REVIEW)